# Dissipative Forces

galpy supports velocity-dependent (dissipative) forces that can be combined
with regular gravitational potentials for orbit integration. These are
implemented as subclasses of `DissipativeForce`.

**Important:** Dissipative forces only work in 3D orbit integration.

In [ ]:
%matplotlib inline
import numpy
from matplotlib import pyplot as plt
from galpy import potential
from galpy.potential import (
    MWPotential2014,
    LogarithmicHaloPotential,
    ChandrasekharDynamicalFrictionForce,
)
from galpy.orbit import Orbit

## Chandrasekhar Dynamical Friction

`ChandrasekharDynamicalFrictionForce` implements the classical Chandrasekhar
dynamical-friction formula. It requires a background potential (whose density
is used to compute the local density) and the mass of the sinking object.

In [ ]:
from astropy import units as u

# Logarithmic halo as the host
lp = LogarithmicHaloPotential(normalize=1.0, q=1.0)

# Dynamical friction from a satellite of mass 5e10 Msun
cdf = ChandrasekharDynamicalFrictionForce(
    GMs=0.5,  # satellite mass in natural units (fraction of v_0^2 * R_0)
    rhm=0.125,  # half-mass radius of the satellite
    dens=lp,  # background density from this potential
)

print(type(cdf))

In [ ]:
# Integrate an orbit with and without dynamical friction
o_nodf = Orbit([1.5, 0.1, 0.8, 0.0, 0.0, 0.0])
o_df = Orbit([1.5, 0.1, 0.8, 0.0, 0.0, 0.0])

ts = numpy.linspace(0.0, 30.0, 10001)

# Without friction
o_nodf.integrate(ts, lp)
# With friction: combine potential + dissipative force
o_df.integrate(ts, lp + cdf)

In [ ]:
# Compare the orbital radius over time
plt.plot(ts, o_nodf.R(ts), label="No friction")
plt.plot(ts, o_df.R(ts), label="With dynamical friction")
plt.xlabel(r"$t$")
plt.ylabel(r"$R / R_0$")
plt.legend()
plt.title("Effect of dynamical friction on orbital decay")

In [ ]:
# Plot the orbit in x-y
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(o_nodf.x(ts), o_nodf.y(ts))
axes[0].set_title("No friction")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].set_aspect("equal")
axes[1].plot(o_df.x(ts), o_df.y(ts))
axes[1].set_title("With dynamical friction")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_aspect("equal")
plt.tight_layout()

## Energy dissipation

We can verify that dynamical friction removes energy from the orbit:

In [ ]:
plt.plot(ts, o_nodf.E(ts, pot=lp), label="No friction")
plt.plot(ts, o_df.E(ts, pot=lp), label="With friction")
plt.xlabel(r"$t$")
plt.ylabel(r"$E$")
plt.legend()
plt.title("Energy evolution")

## NonInertialFrameForce

galpy also provides `NonInertialFrameForce` for integrating orbits in
non-inertial (e.g., rotating or accelerating) reference frames. This is
covered in detail in the **Orbits** tutorial section, since it is most
naturally used in the context of orbit integration.

Key points:
- It adds fictitious forces (Coriolis, centrifugal, etc.)
- Specified via `Omega` (rotation) and/or `a0` (linear acceleration)
- 3D only, like all dissipative forces